## 1. 데이터 선택
- `price`를 뺀 나머지 열을 특징(X), `price`를 정답(y)으로 나눈다

In [7]:
import pandas as pd
# csv파일에서 데이터를 읽어서 표로 만듬
df = pd.read_csv("used_cars.csv", encoding="utf-8")
# print(df.head()) # 잘 읽어왔는지 확인(위 5줄만)

# 문제(특징. feature). 특징 선택할 땐 반드시 리스트로 []
X = df[["car_type", "year_age", "mileage_km", "engine_cc", "accident"]]
# print(X.head()) # 확인
# 답(레이벨. label)
y = df["price"]
# print(y.head())

## 2. 사전 준비

### 차종 원-핫 인코딩

- 차종 목록을 `["경차", "세단", "SUV"]`로 정해 둔다
- 차종을 원-핫 인코딩해서 0/1 열로 바꾼다
- 이번에는 차종 열 3개를 모두 쓴다 (`drop_first`를 쓰지 않는다). KNN은 거리로 비교하기 때문에 하나를 빼면 빠진 차종만 다른 차종과의 거리가 달라진다
- 하나를 빼는 것은 선형회귀일 때다 (집값 원-핫 인코딩 문서)

### 열마다 숫자 크기 보기

- 열마다 최솟값, 최댓값, 표준편차를 확인한다

확인할 것

- 차종 하나가 몇 개의 열이 되었는가?
- 숫자 크기가 가장 큰 열은 무엇인가? 이대로 거리를 재면 어떤 열이 결과를 좌우할까?
- 스케일링은 왜 이 단계가 아니라 분리 뒤에 하는가?

In [12]:
# 원핫인코딩

CAR_TYPE = ["경차", "세단", "SUV"] 

X2 = X.copy()
X2["car_type"] = pd.Categorical(X2["car_type"], categories=CAR_TYPE)
X2 = pd.get_dummies(X2, columns=["car_type"], dtype=int)
# X2.head()

# 스케일링 전 각 특징의 최소, 최대, 표준편차
df.describe().loc[["min", "max", "std"]].round(1)
# 스케일링이 필요. 
# => 사용할 알고리즘이 거리 계산이 필요한 알고리즘인데 안하면 주행거리에 영향을 많이 받기 때문에

,year_age,mileage_km,engine_cc,accident,price
min,0.0,5000.0,1000.0,0.0,200.0
max,15.0,200000.0,3500.0,1.0,4590.0
std,4.7,58512.8,796.4,0.4,1156.9


## 3. 훈련, 테스트 분리

- 학습용 80%, 시험용 20%로 나눈다
- `random_state=42`를 넣어 매번 같은 방식으로 나누게 한다

In [18]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X2, y, test_size=0.2, random_state=42)

## 4. 학습

### 스케일링 없이 KNN

- 이웃 5개로 KNN을 학습한다

### 스케일링 후 KNN

- 학습용 데이터로 스케일러의 기준을 정하고 변환한다 (`fit_transform`)
- 시험용 데이터는 같은 기준으로 변환만 한다 (`transform`)
- 스케일링한 데이터로 KNN을 다시 학습한다

확인할 것

- 변환한 데이터의 최솟값, 최댓값, 표준편차를 2단계와 비교한다
- 원-핫으로 만든 차종 열도 함께 스케일링되었는가?

In [19]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler

# KNN, 스케일링 안 함
knn = KNeighborsRegressor(n_neighbors=5).fit(X_train, y_train)

# 스케일링: 기준은 학습용 데이터로 정하고, 시험용은 같은 기준으로 바꾸기만 한다
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# KNN, 스케일링 함
knn_s = KNeighborsRegressor(n_neighbors=5).fit(X_train_s, y_train)

## 5. 평가

- 두 모델의 평균 오차(만 원)를 학습용, 시험용 데이터로 각각 출력한다

확인할 것

- 스케일링으로 오차가 얼마나 줄었는가? 왜 줄었는지 한 줄로 써 본다
- 학습 오차와 시험 오차의 차이는 어떤가?

In [23]:
# 예측한 중고차 가격과 실제 중고차 가격의 가격 차이
print(f"KNN 스케일링 X: {mean_absolute_error(y_test, knn.predict(X_test)):.0f}만원")
print(f"KNN 스케일링 O: {mean_absolute_error(y_test, knn_s.predict(X_test_s)):.0f}만원")

KNN 스케일링 X: 780만원
KNN 스케일링 O: 136만원


## 6. 예측

- 예측할 차: SUV, 출고 3년, 주행거리 40,000km, 배기량 2,500cc, 사고 없음
- 학습 때와 같은 순서로 바꾼다: 원-핫 인코딩(2단계) → 스케일링(4단계)
- 원-핫은 2단계의 차종 목록을 그대로 쓴다. 차가 한 대뿐이어도 차종 열 3개가 모두 만들어져야 한다
- 스케일링은 새로 `fit`하지 않고, 4단계에서 만든 스케일러로 `transform`만 한다

In [29]:
# print(X.columns)
car_type = ["SUV"]
year_age = [3]
mileage_km = [40000]
engine_cc = [2500]
accident = [0]
new_used_cars = pd.DataFrame({ 
    "car_type" : car_type, "year_age" : year_age, "mileage_km" : mileage_km, 
    "engine_cc" : engine_cc, "accident" : accident
})

# 원핫 인코딩
new_used_cars["car_type"] = pd.Categorical(new_used_cars["car_type"], categories=CAR_TYPE)
new_used_cars = pd.get_dummies(new_used_cars, columns=["car_type"], dtype=int)

# 스케일링
new_used_cars = scaler.transform(new_used_cars)

# 예측
new_predict = knn_s.predict(new_used_cars)
# SUV, 출고 3년, 주행거리 40,000km, 배기량 2,500cc, 사고 없음
print(f"차종 : {car_type[0]}, 연식 : {year_age[0]}, 주행거리 : {mileage_km[0]}km, ", end="")
print(f"배기량 : {engine_cc[0]}cc, 사고유무 : {"있음" if accident[0] else "없음"}")
print(f"예측 시세 : {new_predict[0]:.0f}만원")

차종 : SUV, 연식 : 3, 주행거리 : 40000km, 배기량 : 2500cc, 사고유무 : 없음
예측 시세 : 3202만원
